# VLA01：RGB/text/state → tokens → action field

用小型随机 Transformer 追一次完整 VLA tensor、mask、loss 与 sampler；不下载模型，不进行本地/云端预训练复现。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


## 2. 合成 observation/action contract
2相机、6个text slots、1个state token、8个未来动作。动作是7个合成连续坐标，没有真实机器人控制语义。部分text/action slots无效。


In [ ]:
BATCH, CAMERAS, IMAGE_SIZE, PATCH = 2, 2, 32, 8
TEXT_LENGTH, HORIZON, ACTION_DIM, WIDTH = 6, 8, 7, 32
images = torch.rand(BATCH, CAMERAS, 3, IMAGE_SIZE, IMAGE_SIZE)
text_ids = torch.randint(1, 64, (BATCH, TEXT_LENGTH))
text_valid = torch.ones(BATCH, TEXT_LENGTH, dtype=torch.bool)
text_valid[1, -2:] = False
proprio = torch.randn(BATCH, 7)
coordinate_scale = torch.linspace(.2, 1., ACTION_DIM)
synthetic_train_actions = torch.randn(24, HORIZON, ACTION_DIM)*coordinate_scale
train_mean = synthetic_train_actions.mean(dim=(0, 1))
train_std = synthetic_train_actions.std(dim=(0, 1), correction=0).clamp_min(1e-6)
clean_coordinates = torch.randn(BATCH, HORIZON, ACTION_DIM)*coordinate_scale
clean_actions = (clean_coordinates-train_mean)/train_std
action_valid = torch.ones(BATCH, HORIZON, dtype=torch.bool)
action_valid[1, -2:] = False
noise = torch.randn_like(clean_actions)
tau = torch.rand(BATCH, 1, 1)
noisy_actions = (1-tau)*noise + tau*clean_actions
velocity_target = clean_actions-noise
print('RGB:', images.shape, '| text:', text_ids.shape, '| state:', proprio.shape)
print('clean/noisy/target:', clean_actions.shape, noisy_actions.shape, velocity_target.shape)


## 3. 三条输入支路与 P/S/A block mask
这是共享小Transformer，不是π0的多expert实现。P读P；S读P/S；A读P/S/A。Noisy A 可双向读取，future sensor不能因此进入条件。


In [ ]:
class TinyVLATrace(nn.Module):
    def __init__(self):
        super().__init__()
        patches = (IMAGE_SIZE//PATCH)**2
        self.patch = nn.Conv2d(3, WIDTH, kernel_size=PATCH, stride=PATCH)
        self.camera = nn.Parameter(.02*torch.randn(1, CAMERAS, 1, WIDTH))
        self.patch_position = nn.Parameter(.02*torch.randn(1, 1, patches, WIDTH))
        self.text = nn.Embedding(64, WIDTH)
        self.text_position = nn.Parameter(.02*torch.randn(1, TEXT_LENGTH, WIDTH))
        self.state = nn.Linear(7, WIDTH)
        self.action = nn.Linear(ACTION_DIM, WIDTH)
        self.action_position = nn.Parameter(.02*torch.randn(1, HORIZON, WIDTH))
        self.time = nn.Sequential(nn.Linear(1, WIDTH), nn.SiLU(), nn.Linear(WIDTH, WIDTH))
        layer = nn.TransformerEncoderLayer(WIDTH, 4, dim_feedforward=64, dropout=0., batch_first=True)
        self.backbone = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.output = nn.Linear(WIDTH, ACTION_DIM)
    def forward(self, rgb, ids, text_mask, state, noisy, generation_time):
        b, cameras = rgb.shape[:2]
        patches = self.patch(rgb.flatten(0, 1)).flatten(2).transpose(1, 2)
        patches = patches.reshape(b, cameras, -1, WIDTH)
        image_tokens = (patches+self.camera+self.patch_position).flatten(1, 2)
        text_tokens = self.text(ids)+self.text_position
        prefix = torch.cat([image_tokens, text_tokens], dim=1)
        state_token = self.state(state).unsqueeze(1)
        action_tokens = (self.action(noisy)+self.action_position+
                         self.time(generation_time.reshape(b, 1)).unsqueeze(1))
        n_prefix = prefix.shape[1]; total = n_prefix+1+HORIZON
        dependency = torch.zeros(total, total, dtype=torch.bool)
        dependency[:n_prefix, n_prefix:] = True
        dependency[n_prefix:n_prefix+1, n_prefix+1:] = True
        key_valid = torch.cat([torch.ones(b, image_tokens.shape[1], dtype=torch.bool), text_mask,
                               torch.ones(b, 1+HORIZON, dtype=torch.bool)], dim=1)
        hidden = self.backbone(torch.cat([prefix, state_token, action_tokens], dim=1),
                               mask=dependency, src_key_padding_mask=~key_valid)
        velocity = self.output(hidden[:, -HORIZON:])
        shapes = {'image_tokens': list(image_tokens.shape), 'text_tokens': list(text_tokens.shape),
                  'prefix': list(prefix.shape), 'state_token': list(state_token.shape),
                  'action_tokens': list(action_tokens.shape), 'hidden': list(hidden.shape),
                  'velocity': list(velocity.shape), 'dependency_mask': list(dependency.shape)}
        return velocity, hidden[:, :n_prefix], dependency, shapes
model = TinyVLATrace()
predicted, _, dependency, shapes = model(images, text_ids, text_valid, proprio, noisy_actions, tau)
display(pd.DataFrame({'shape': shapes}).T)
assert predicted.shape == clean_actions.shape
plt.imshow((~dependency).numpy(), vmin=0, vmax=1)
plt.xlabel('key'); plt.ylabel('query'); plt.title('Allowed P/S/A dependencies'); plt.show()


## 4. Masked loss 与一次 backward
不是训练成功记录；只检查模型收到的目标、分母和gradient。Invalid target不计loss，不默认把loss mask用作suffix attention mask。


In [ ]:
model.zero_grad(set_to_none=True)
valid_scalar = action_valid.unsqueeze(-1).expand_as(predicted)
assert int(valid_scalar.sum()) > 0
objective = ((predicted-velocity_target).square()*valid_scalar).sum()/valid_scalar.sum()
objective.backward()
gradients = {name: float(parameter.grad.norm()) for name, parameter in model.named_parameters()
             if parameter.grad is not None}
assert all(math.isfinite(value) for value in gradients.values())
print('masked flow loss:', float(objective.detach()), '| valid scalars:', int(valid_scalar.sum()))
print('vision gradient norm:', gradients['patch.weight'])


## 5. 多层信息依赖的 perturbation checks
No dropout；改变noisy actions不应改prefix。改变无效text padding不应改有效action outputs。此检查说明cache依赖条件，不实现KV cache。


In [ ]:
model.eval()
with torch.no_grad():
    original_v, original_prefix, _, _ = model(images, text_ids, text_valid, proprio, noisy_actions, tau)
    _, changed_prefix, _, _ = model(images, text_ids, text_valid, proprio, noisy_actions+100., tau)
    changed_text = text_ids.clone(); changed_text[~text_valid] = 63
    changed_v, _, _, _ = model(images, changed_text, text_valid, proprio, noisy_actions, tau)
    prefix_delta = float((original_prefix-changed_prefix).abs().max())
    pad_delta = float((original_v-changed_v).abs().max())
assert prefix_delta < 1e-5 and pad_delta < 1e-5
print('prefix delta:', prefix_delta, '| text-padding action delta:', pad_delta)


## 6. Noise→action sampler 与反归一化trace
随机网络的输出没有任务能力。Sampler不接收clean expert endpoint；为可读性每步仍重算全部小网络。统计来自独立synthetic train数据。


In [ ]:
@torch.no_grad()
def sample_chunk(model, rgb, ids, valid_text, state, steps=4):
    model.eval(); chunk = torch.randn(len(rgb), HORIZON, ACTION_DIM)
    for index in range(steps):
        time = torch.full((len(rgb), 1, 1), index/steps)
        vector_field, _, _, _ = model(rgb, ids, valid_text, state, chunk, time)
        chunk = chunk+vector_field/steps
    return chunk
recovered = clean_actions*train_std+train_mean
assert torch.allclose(recovered, clean_coordinates, atol=1e-6)
sampled = sample_chunk(model, images, text_ids, text_valid, proprio)
output_coordinates = sampled*train_std+train_mean
results = {'seed': SEED, 'shapes': shapes, 'masked_loss_at_random_initialization': float(objective.detach()),
           'prefix_perturbation_delta': prefix_delta, 'padding_perturbation_delta': pad_delta,
           'sample_shape': list(output_coordinates.shape), 'optimizer_updates': 0,
           'scope': 'Random-weight tensor/gradient/mask trace. No pretrained VLA, robot command, or performance validation.'}
print(json.dumps(results, indent=2))


## 面试追问与实验记录
1. 计算prefix为何有38个tokens，总序列为何47。
2. 为什么suffix未来位置可双向读，但未来真实force不可读？
3. 原生action、模型action width、hidden width有何区别？
4. 解释why cache independence is not an implemented KV cache。
5. 把camera数改为1需要同时改哪些接口？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
